# CancerDetector

In [1]:
import sys
from pathlib import Path
import json

import tempfile

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from scipy import stats
from tqdm.notebook import tqdm as tqdm_notebook

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

from methyldl.modelling.classifiers.cancer_detector import CancerDetectorClassifier

%load_ext autoreload
%autoreload 2

## Load the data

In [2]:
with open(ROOT /"App" / "labels_dict.json", "r") as f:
    label_dict = json.load(f)
cell_types_names = sorted(list(label_dict.values()))
del label_dict
cell_labels_to_names = {i: n for i, n in enumerate(cell_types_names)}
cell_names_to_labels = {n: i for i, n in enumerate(cell_types_names)}

In [3]:
DATA_PATH = ROOT / "Data" / "training_data" / "SoftLabelsTrainingData_205files_pooled_Jaccard_hg38_mincpg_4_minlen_10_no_data_leak_d041"
train_data = pd.read_parquet(DATA_PATH / "train.parquet")
val_data = pd.read_parquet(DATA_PATH / "valid.parquet")
test_data = pd.read_parquet(DATA_PATH / "test.parquet")

print("Train data:", len(train_data))
print("Validation data:", len(val_data))
print("Test data:", len(test_data))

Train data: 2910114
Validation data: 915902
Test data: 900610


In [ ]:
train_on_target_mask = train_data["original_label"] == train_data["dmr_ctype_label"]
val_on_target_mask = val_data["original_label"] == val_data["dmr_ctype_label"]
test_on_target_mask = test_data["original_label"] == test_data["dmr_ctype_label"]

### Data exploration

On veut quantifier le nombre de dmr qui ont au moins une paire (dmr, classe) avec k échantillons pour k = 1, ..., 10

In [ ]:
results = []
col_dmr_label = "dmr_label"
total_dmrs = train_data[col_dmr_label].nunique()
total_reads = len(train_data)
for k in tqdm_notebook(range(1, 5)):
    # on calcule le nombre de dmr telles que toutes les paires (dmr, classe) ont au moins k échantillons
    n_dmr = train_data.groupby(col_dmr_label).filter(lambda x: (x["original_label"].value_counts() >= k).all())[col_dmr_label].nunique()
    # on calcule le nombre d'échantillons restants
    n_reads = train_data.groupby(col_dmr_label).filter(lambda x: (x["original_label"].value_counts() >= k).all()).shape[0]

    results.append({
        "k": k,
        "n_dmr": n_dmr,
        "n_reads": n_reads,
        "pct_dmr": n_dmr / total_dmrs,
        "pct_reads": n_reads / total_reads
    })
pd.DataFrame(results)

## Train and evaluate CancerDetector

### Demo with class prior estimated from training data frequencies

In [ ]:
cd_clf = CancerDetectorClassifier()
cd_clf.fit(
    train_data,
    col_n_meth_cpgs="M",
    col_n_unmeth_cpgs="U",
    col_marker_label="dmr_ctype_label",
    eps_beta_fit=0.01,
    class_prior_type="train_freq",
    # verbose=True,
)

In [ ]:
train_probas, train_lhs = cd_clf.predict_proba(
    train_data, return_likelihoods=True, verbose=True
)
val_probas, val_lhs = cd_clf.predict_proba(
    val_data, return_likelihoods=True, verbose=True
)
test_probas, test_lhs = cd_clf.predict_proba(
    test_data, return_likelihoods=True, verbose=True
)

In [ ]:
# compute predicted labels and ground truth labels for train, val and test sets
train_preds = np.argmax(train_probas, axis=1)
train_gt = train_data["original_label"].to_numpy()
val_preds = np.argmax(val_probas, axis=1)
val_gt = val_data["original_label"].to_numpy()
test_preds = np.argmax(test_probas, axis=1)
test_gt = test_data["original_label"].to_numpy()

In [ ]:
train_clf_report = classification_report(train_gt, train_preds, target_names=cell_types_names)
val_clf_report = classification_report(val_gt, val_preds, target_names=cell_types_names)
test_clf_report = classification_report(test_gt, test_preds, target_names=cell_types_names)

In [ ]:
print("Train classification report:\n", train_clf_report)

In [ ]:
print(f"Val classification report:\n{val_clf_report}")

In [ ]:
# compute class-wise accuracy for train, val and test sets
def compute_class_wise_acc(y_true, y_pred, class_labels):
    acc_per_class = {}
    for label in class_labels:
        idx = y_true == label
        acc = accuracy_score(y_true[idx], y_pred[idx])
        acc_per_class[cell_labels_to_names[label]] = acc
    return acc_per_class

train_cw_acc = compute_class_wise_acc(train_gt, train_preds, class_labels=cell_names_to_labels.values())
val_cw_acc = compute_class_wise_acc(val_gt, val_preds, class_labels=cell_names_to_labels.values())
test_cw_acc = compute_class_wise_acc(test_gt, test_preds, class_labels=cell_names_to_labels.values())

pd.DataFrame({
    "train": train_cw_acc,
    "validation": val_cw_acc,
    "test": test_cw_acc,
})

In [ ]:
# confusion matrices
train_cm = confusion_matrix(train_gt, train_preds)
val_cm = confusion_matrix(val_gt, val_preds)
test_cm = confusion_matrix(test_gt, test_preds)

def plot_cm(cm, class_names, title, row_normalization=False):
    if row_normalization:
        cm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
    plt.figure(figsize=(10, 8))
    plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=90)
    plt.yticks(tick_marks, class_names)
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.show()

plot_cm(train_cm, class_names=cell_types_names, title="Train Confusion Matrix", row_normalization=True)

In [ ]:
train_lhs[~train_on_target_mask][0]

In [ ]:
# confusion matrices for on-target reads only
train_cm_on_target = confusion_matrix(train_gt[train_on_target_mask], train_preds[train_on_target_mask])
val_cm_on_target = confusion_matrix(val_gt[val_on_target_mask], val_preds[val_on_target_mask])
test_cm_on_target = confusion_matrix(test_gt[test_on_target_mask], test_preds[test_on_target_mask])

plot_cm(train_cm_on_target, class_names=cell_types_names, title="Train Confusion Matrix (On-target reads)", row_normalization=True)

In [ ]:
train_cnt_per_cell_type = train_data.groupby("original_label").agg(count = ("original_label", "count")).reset_index()
train_cnt_per_cell_type["cell_type"] = train_cnt_per_cell_type["original_label"].map(cell_labels_to_names)
train_cnt_per_cell_type.sort_values("count", ascending=False)

It seems that the class prior pushes the less frequent classes to be predicted less often.

In [ ]:
# correlation between the ratio (FN / (FN + TP)) (ie 1-recall, or missed detections) and the class prior
train_cm_norm = train_cm.astype("float") / train_cm.sum(axis=1)[:, np.newaxis]
class_priors = np.bincount(train_gt) / len(train_gt)
missed_detections = 1 - np.diag(train_cm_norm)
corr = np.corrcoef(class_priors, missed_detections)[0, 1]
plt.figure(figsize=(8, 6))
plt.scatter(class_priors, missed_detections)
plt.ylim(0, 1.05)
plt.xlabel("Class Prior")
plt.ylabel("Missed Detections (1 - Recall)")
plt.title(f"Train Correlation between Class Prior and Missed Detections ({corr:.2f})")
plt.grid(True)
plt.show()

### Demo with uniform class prior

In [4]:
cd_clf_unif = CancerDetectorClassifier()
cd_clf_unif.fit(
    train_data,
    col_n_meth_cpgs="M",
    col_n_unmeth_cpgs="U",
    col_marker_label="dmr_ctype_label",
    eps_beta_fit=0.01,
    class_prior_type="uniform",
    # verbose=True,
)

In [5]:
save_model_path = ROOT / "output" / "cancer_detector" / f"CD_trainlen={len(train_data)}_epsbetafit={0.01}_prior=uniform.pkl"
cd_clf_unif.save(save_model_path)
print(f"CancerDetector model saved to: {save_model_path}")

CancerDetector model saved to: /home/nathan/Documents/Scolaire/7_Cesure/2_KU_Leuven/methyldl/output/cancer_detector/CD_trainlen=2910114_epsbetafit=0.01_prior=uniform.pkl


In [ ]:
train_probas_unif, train_lhs_unif = cd_clf_unif.predict_proba(
    train_data, return_likelihoods=True, verbose=True
)
val_probas_unif, val_lhs_unif = cd_clf_unif.predict_proba(
    val_data, return_likelihoods=True, verbose=True
)
test_probas_unif, test_lhs_unif = cd_clf_unif.predict_proba(
    test_data, return_likelihoods=True, verbose=True
)

In [ ]:
# compute predicted labels and ground truth labels for train, val and test sets
train_preds_unif = np.argmax(train_probas_unif, axis=1)
train_gt = train_data["original_label"].to_numpy()
val_preds_unif = np.argmax(val_probas_unif, axis=1)
val_gt = val_data["original_label"].to_numpy()
test_preds_unif = np.argmax(test_probas_unif, axis=1)
test_gt = test_data["original_label"].to_numpy()

In [ ]:
# compute classification reports and confusion matrices for the uniform class prior model
train_clf_report_unif = classification_report(train_gt, train_preds_unif, target_names=cell_types_names)
val_clf_report_unif = classification_report(val_gt, val_preds_unif, target_names=cell_types_names)
test_clf_report_unif = classification_report(test_gt, test_preds_unif, target_names=cell_types_names)

train_cm_unif = confusion_matrix(train_gt, train_preds_unif)
val_cm_unif = confusion_matrix(val_gt, val_preds_unif)
test_cm_unif = confusion_matrix(test_gt, test_preds_unif)

train_cw_acc_unif = compute_class_wise_acc(train_gt, train_preds_unif, class_labels=cell_names_to_labels.values())
val_cw_acc_unif = compute_class_wise_acc(val_gt, val_preds_unif, class_labels=cell_names_to_labels.values())
test_cw_acc_unif = compute_class_wise_acc(test_gt, test_preds_unif, class_labels=cell_names_to_labels.values())
cw_acc_df_unif = pd.DataFrame({
    "train": train_cw_acc_unif,
    "validation": val_cw_acc_unif,
    "test": test_cw_acc_unif,
})

In [ ]:
print(train_clf_report_unif)

In [ ]:
cw_acc_df_unif

In [ ]:
plot_cm(train_cm_unif, class_names=cell_types_names, title="Train Confusion Matrix (Uniform Prior)", row_normalization=True)

In [ ]:
# plot the beta distributions 
marker_label = 6
marker_ctype_name = cell_labels_to_names[marker_label]
eta_per_ctype = cd_clf_unif.param_eta_matrix[marker_label, :]
rho_per_ctype = cd_clf_unif.param_rho_matrix[marker_label, :]

x = np.linspace(0, 1, 100)
plt.figure(figsize=(10, 10))
for ctype_label, ctype_name in cell_labels_to_names.items():
    eta = eta_per_ctype[ctype_label]
    rho = rho_per_ctype[ctype_label]
    y = stats.beta.pdf(x, eta, rho)
    if ctype_label == marker_label:
        plt.plot(x, y, label=f"{ctype_name} (marker)", linewidth=3, color="red")
    else:
        plt.plot(x, y, label=ctype_name)
plt.title(f"Beta Distributions for Marker {marker_label} ({marker_ctype_name})")
plt.xlabel("Beta Value")
plt.ylabel("Density")
plt.ylim(0, 10)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(True)
plt.show()

## Train CancerDetector Classifier and generate ios

ios are input/outputs of deconvolvers (input: prediction matricess, output: cell type proportions)